## Credit Card Kaggle- Handle Imbalanced Dataset
#### Context
It is important that credit card companies are able to recognize fraudulent credit card transactions so that customers are not charged for items that they did not purchase.



#### Inspiration
Identify fraudulent credit card transactions.

Given the class imbalance ratio, we recommend measuring the accuracy using the Area Under the Precision-Recall Curve (AUPRC). Confusion matrix accuracy is not meaningful for unbalanced classification.



In [24]:
import numpy as np
import pandas as pd
import sklearn
import scipy
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report,accuracy_score
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
# Sets a global default plot size (14 inches wide, 8 inches tall) for all matplotlib plots in this notebook —
from pylab import rcParams
rcParams['figure.figsize'] = 14, 8

RANDOM_SEED = 42
LABELS = ["Normal", "Fraud"]

In [25]:
data = pd.read_csv('credit_dataset.csv',sep=',' ,index_col=0)
print(data.shape)
data.head()

(25134, 19)


,ID,GENDER,CAR,REALITY,NO_OF_CHILD,INCOME,INCOME_TYPE,EDUCATION_TYPE,FAMILY_TYPE,HOUSE_TYPE,FLAG_MOBIL,WORK_PHONE,PHONE,E_MAIL,FAMILY SIZE,BEGIN_MONTH,AGE,YEARS_EMPLOYED,TARGET
0,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,1,0,0,0,2.0,29,59,3,0
1,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,1.0,4,52,8,0
2,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,1.0,26,52,8,0
3,5008810,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,1.0,26,52,8,0
4,5008811,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,1.0,38,52,8,0


In [26]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25134 entries, 0 to 25133
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              25134 non-null  int64  
 1   GENDER          25134 non-null  object 
 2   CAR             25134 non-null  object 
 3   REALITY         25134 non-null  object 
 4   NO_OF_CHILD     25134 non-null  int64  
 5   INCOME          25134 non-null  float64
 6   INCOME_TYPE     25134 non-null  object 
 7   EDUCATION_TYPE  25134 non-null  object 
 8   FAMILY_TYPE     25134 non-null  object 
 9   HOUSE_TYPE      25134 non-null  object 
 10  FLAG_MOBIL      25134 non-null  int64  
 11  WORK_PHONE      25134 non-null  int64  
 12  PHONE           25134 non-null  int64  
 13  E_MAIL          25134 non-null  int64  
 14  FAMILY SIZE     25134 non-null  float64
 15  BEGIN_MONTH     25134 non-null  int64  
 16  AGE             25134 non-null  int64  
 17  YEARS_EMPLOYED  25134 non-null  int6

 Before training any model, every object/text column must be encoded into numbers 

In [27]:
# check which specific columns are object type
data.select_dtypes(include='object').columns

Index(['GENDER', 'CAR', 'REALITY', 'INCOME_TYPE', 'EDUCATION_TYPE',
       'FAMILY_TYPE', 'HOUSE_TYPE'],
      dtype='object')

In [28]:
categorical_features = [feature for feature in data.columns if data[feature].dtypes == 'O']
print("No of Categorical variables:", len(categorical_features))
data[categorical_features].head()

No of Categorical variables: 7


,GENDER,CAR,REALITY,INCOME_TYPE,EDUCATION_TYPE,FAMILY_TYPE,HOUSE_TYPE
0,M,Y,Y,Working,Secondary / secondary special,Married,House / apartment
1,F,N,Y,Commercial associate,Secondary / secondary special,Single / not married,House / apartment
2,F,N,Y,Commercial associate,Secondary / secondary special,Single / not married,House / apartment
3,F,N,Y,Commercial associate,Secondary / secondary special,Single / not married,House / apartment
4,F,N,Y,Commercial associate,Secondary / secondary special,Single / not married,House / apartment


#### This confirms which are binary (2 categories → Label Encoding) vs multi-category (3+ → One-Hot).

In [29]:
cat_cols = ['GENDER', 'CAR', 'REALITY', 'INCOME_TYPE', 'EDUCATION_TYPE', 'FAMILY_TYPE', 'HOUSE_TYPE']

for col in cat_cols:
    print(col, '→', data[col].unique(), '| Total categories:', data[col].nunique())

GENDER → ['M' 'F'] | Total categories: 2
CAR → ['Y' 'N'] | Total categories: 2
REALITY → ['Y' 'N'] | Total categories: 2
INCOME_TYPE → ['Working' 'Commercial associate' 'State servant' 'Student' 'Pensioner'] | Total categories: 5
EDUCATION_TYPE → ['Secondary / secondary special' 'Higher education' 'Incomplete higher'
 'Lower secondary' 'Academic degree'] | Total categories: 5
FAMILY_TYPE → ['Married' 'Single / not married' 'Civil marriage' 'Separated' 'Widow'] | Total categories: 5
HOUSE_TYPE → ['House / apartment' 'Rented apartment' 'Municipal apartment'
 'With parents' 'Co-op apartment' 'Office apartment'] | Total categories: 6


#### Label Encoding — for binary columns

In [30]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

data['GENDER'] = le.fit_transform(data['GENDER'])
data['CAR'] = le.fit_transform(data['CAR'])
data['REALITY'] = le.fit_transform(data['REALITY'])

What happens: Each column's 2 unique text values get converted to 0 and 1 (alphabetical order by default — e.g., 'F'→0, 'M'→1).

In [31]:
print(data['GENDER'].unique())   # should show [0, 1] now
print(data['CAR'].unique())
print(data['REALITY'].unique())

[1 0]
[1 0]
[1 0]


#### One-Hot Encoding — for multi-category columns

In [32]:
data = pd.get_dummies(data, columns=['INCOME_TYPE', 'EDUCATION_TYPE', 'FAMILY_TYPE', 'HOUSE_TYPE'], drop_first=True)

What happens: Each category becomes its own new 0/1 column. drop_first=True drops one category per original column (avoids redundancy — if all other dummy columns are 0, that dropped one is implied).

Before:

INCOME_TYPE: ['Working', 'Commercial', 'Pensioner']

After
(drop_first=True, 'Commercial' dropped as baseline):
INCOME_TYPE_Pensioner | INCOME_TYPE_Working
0                      | 1        → Working
0                      | 0        → Commercial (baseline, implied)
1                      | 0        → Pensioner

In [33]:
data.info()
print(data.select_dtypes(include='object').columns)   # should print: Index([], dtype='object')

<class 'pandas.core.frame.DataFrame'>
Index: 25134 entries, 0 to 25133
Data columns (total 32 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   ID                                            25134 non-null  int64  
 1   GENDER                                        25134 non-null  int64  
 2   CAR                                           25134 non-null  int64  
 3   REALITY                                       25134 non-null  int64  
 4   NO_OF_CHILD                                   25134 non-null  int64  
 5   INCOME                                        25134 non-null  float64
 6   FLAG_MOBIL                                    25134 non-null  int64  
 7   WORK_PHONE                                    25134 non-null  int64  
 8   PHONE                                         25134 non-null  int64  
 9   E_MAIL                                        25134 non-null  int6

In [34]:
print(data.shape)
data.head()

(25134, 32)


,ID,GENDER,CAR,REALITY,NO_OF_CHILD,INCOME,FLAG_MOBIL,WORK_PHONE,PHONE,E_MAIL,...,EDUCATION_TYPE_Secondary / secondary special,FAMILY_TYPE_Married,FAMILY_TYPE_Separated,FAMILY_TYPE_Single / not married,FAMILY_TYPE_Widow,HOUSE_TYPE_House / apartment,HOUSE_TYPE_Municipal apartment,HOUSE_TYPE_Office apartment,HOUSE_TYPE_Rented apartment,HOUSE_TYPE_With parents
0,5008806,1,1,1,0,112500.0,1,0,0,0,...,True,True,False,False,False,True,False,False,False,False
1,5008808,0,0,1,0,270000.0,1,0,1,1,...,True,False,False,True,False,True,False,False,False,False
2,5008809,0,0,1,0,270000.0,1,0,1,1,...,True,False,False,True,False,True,False,False,False,False
3,5008810,0,0,1,0,270000.0,1,0,1,1,...,True,False,False,True,False,True,False,False,False,False
4,5008811,0,0,1,0,270000.0,1,0,1,1,...,True,False,False,True,False,True,False,False,False,False


In [35]:
data.info()
data.shape

<class 'pandas.core.frame.DataFrame'>
Index: 25134 entries, 0 to 25133
Data columns (total 32 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   ID                                            25134 non-null  int64  
 1   GENDER                                        25134 non-null  int64  
 2   CAR                                           25134 non-null  int64  
 3   REALITY                                       25134 non-null  int64  
 4   NO_OF_CHILD                                   25134 non-null  int64  
 5   INCOME                                        25134 non-null  float64
 6   FLAG_MOBIL                                    25134 non-null  int64  
 7   WORK_PHONE                                    25134 non-null  int64  
 8   PHONE                                         25134 non-null  int64  
 9   E_MAIL                                        25134 non-null  int6

(25134, 32)

One thing worth understanding: the bool columns
You'll notice dtypes: bool(17) — this is fine, but good to know why:
Newer pandas versions create One-Hot columns as bool (True/False) instead of int (0/1) by default. Functionally, this is identical — True behaves as 1, False as 0 in almost all math/model operations. Most sklearn models handle bool columns without any issue.
But — for full safety, especially before SMOTE and scaling, it's a good habit to explicitly convert them to int:

In [38]:
bool_cols = data.select_dtypes(include='bool').columns
data[bool_cols] = data[bool_cols].astype(int)

In [39]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25134 entries, 0 to 25133
Data columns (total 32 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   ID                                            25134 non-null  int64  
 1   GENDER                                        25134 non-null  int64  
 2   CAR                                           25134 non-null  int64  
 3   REALITY                                       25134 non-null  int64  
 4   NO_OF_CHILD                                   25134 non-null  int64  
 5   INCOME                                        25134 non-null  float64
 6   FLAG_MOBIL                                    25134 non-null  int64  
 7   WORK_PHONE                                    25134 non-null  int64  
 8   PHONE                                         25134 non-null  int64  
 9   E_MAIL                                        25134 non-null  int6

In [43]:
data['TARGET'].value_counts()

TARGET
0    24712
1      422
Name: count, dtype: int64

In [44]:
data['TARGET'].value_counts(normalize=True) * 100

TARGET
0    98.320999
1     1.679001
Name: proportion, dtype: float64

Hence shown this is imbaalnce data

#### Split first (train/test) — before any resampling

In [45]:
from sklearn.model_selection import train_test_split

X = data.drop('TARGET', axis=1)
y = data['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#### Apply SMOTE only on training data

In [48]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.3, random_state=42)  # brings minority to ~30% instead of full 50%
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(y_train_resampled.value_counts(normalize=True) * 100)

TARGET
0    76.925172
1    23.074828
Name: proportion, dtype: float64


In [54]:
## Get the Fraud and the normal dataset 

fraud = data[data['TARGET']==1]

normal = data[data['TARGET']==0]

In [55]:
print(fraud.shape,normal.shape)

(422, 32) (24712, 32)


### UNDERSAMPLING & OVERSAMPLING
Undersampling reduces the majority class size until it roughly matches the minority class, so the model sees a balanced ratio during training. NearMiss specifically does this smartly — it keeps only majority-class points closest to the minority class boundary (most useful for distinguishing classes), instead of removing randomly.

In [64]:
from imblearn.under_sampling import NearMiss

# Implementing Undersampling for Handling Imbalanced 
nm = NearMiss()
X_res,y_res=nm.fit_resample(X,y)

In [65]:
X_res.shape,y_res.shape

((844, 31), (844,))

In [67]:
from collections import Counter
print('Original dataset shape {}'.format(Counter(y)))
print('Resampled dataset shape {}'.format(Counter(y_res)))

Original dataset shape Counter({0: 24712, 1: 422})
Resampled dataset shape Counter({0: 422, 1: 422})


Why NearMiss undersampling here is risky
Resampled: {0: 422, 1: 422}
You went from 24,712 majority rows down to just 422 — you threw away 24,290 real data points (98% of your majority class!) just to force balance.

### SMOTE (keeps all your data)(oversampling)
This keeps all 24,712 majority rows, and adds synthetic minority examples up to a reasonable ratio (not full 50:50, since going from 422→24,712 synthetic points would be excessive/unrealistic — moderate sampling_strategy is safer).


In [77]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.3, random_state=42)  # brings minority to ~30%, not full 50%
X_res, y_res = smote.fit_resample(X_train, y_train)

print(y_res.value_counts())

TARGET
0    19769
1     5930
Name: count, dtype: int64


In [69]:
from collections import Counter
print('Original dataset shape {}'.format(Counter(y)))
print('Resampled dataset shape {}'.format(Counter(y_res)))

Original dataset shape Counter({0: 24712, 1: 422})
Resampled dataset shape Counter({0: 19769, 1: 5930})


In [72]:
from imblearn.combine import SMOTETomek
# Implementing Oversampling for Handling Imbalanced 
smk = SMOTETomek(random_state=42)
X_res,y_res=smk.fit_resample(X,y)

In [73]:
X_res.shape,y_res.shape

((48654, 31), (48654,))

In [78]:
# In this example 
# I use SMOTETomek which is a method of imblearn.
# SMOTETomek is a hybrid method
# which uses an under sampling method (Tomek) in with an over sampling method (SMOTE).

#### RandomOverSampler
The simplest oversampling technique — it balances classes by randomly duplicating existing minority class rows (exact copies) until it matches the majority class count. No synthetic/new data is created (unlike SMOTE) — just repeats of what already exists.

In [80]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X, y)

print(y_res.value_counts())

TARGET
0    24712
1    24712
Name: count, dtype: int64


In [81]:
from collections import Counter
print('Original dataset shape {}'.format(Counter(y)))
print('Resampled dataset shape {}'.format(Counter(y_res)))

Original dataset shape Counter({0: 24712, 1: 422})
Resampled dataset shape Counter({0: 24712, 1: 24712})
